# Internal Benchmarking — Modification Detection

This notebook evaluates **all combinations** of DTW parameters × probability
algorithms for detecting RNA modifications, using known *E. coli* rRNA
modification sites as ground truth.

### Method matrix

| Axis | Values | Count |
|------|--------|-------|
| **padding** | 0, 1, 2 | 3 |
| **open boundaries** | (F,F), (T,F), (F,T), (T,T) | 4 |
| **Probability method** | dist_to_ivt, knn, mds_gmm, hier_V2, hier_V3 | 5 |
| **Total configs** | 3 × 4 × 5 | **60** |

### Evaluation

- **Per-read-per-position binary classification** — each `(read, position)` is
  an instance.  At known modification sites, native reads are labelled
  **modified (1)** and IVT reads **unmodified (0)**.  At all other positions,
  every read (native + IVT) is **unmodified (0)**.
- Metrics: **AUROC**, **AUPRC**, **F1 @ optimal threshold**, precision, recall.

### Position offset

f5c eventalign reports the **start** of the 5-mer kmer.  The modification is
at the centre, so we apply `pipeline_position + 3 = biological_position`.

## 0. Setup & Paths

In [ ]:
import logging
import sys
import time
import warnings
from collections import Counter, defaultdict
from itertools import product
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
# Suppress noisy warnings during benchmarking
warnings.filterwarnings("ignore", category=RuntimeWarning)

TESTDATA = Path("../testdata").resolve()
assert TESTDATA.exists(), f"testdata directory not found: {TESTDATA}"

NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"
IVT_BAM      = TESTDATA / "control_0.bam"
IVT_FASTQ    = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5    = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"
REF_FASTA    = TESTDATA / "ref.fa"

for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "\u2705" if path.exists() else "\u274c MISSING"
    print(f"  {status}  {label:15s}  {path}")

OUTPUT_DIR = Path("../output/benchmarking").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

## 1. Known Modifications & Ground Truth

Known *E. coli* rRNA modification sites.  Positions are **biological (1-based)**.
The mapping to pipeline positions is: `pipeline_pos + 3 = biological_pos`.

**Ground truth rule**:
- At known modification sites: **native reads → 1** (modified), **IVT reads → 0**
- At all other positions: **all reads → 0** (unmodified)

In [ ]:
# Position offset: eventalign reports kmer start, modification is at centre
POSITION_OFFSET = 3

# Known E. coli rRNA modifications — biological positions (1-based)
KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Psi) ---
    ("ecoli16S", 516):  ("Psi", "pseudouridine"),
    ("ecoli23S", 746):  ("Psi", "pseudouridine"),
    ("ecoli23S", 955):  ("Psi", "pseudouridine"),
    ("ecoli23S", 1911): ("Psi", "pseudouridine"),
    ("ecoli23S", 1917): ("Psi", "pseudouridine"),
    ("ecoli23S", 2457): ("Psi", "pseudouridine"),
    ("ecoli23S", 2504): ("Psi", "pseudouridine"),
    ("ecoli23S", 2580): ("Psi", "pseudouridine"),
    ("ecoli23S", 2604): ("Psi", "pseudouridine"),
    ("ecoli23S", 2605): ("Psi", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m66A) ---
    ("ecoli16S", 1518): ("m66A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m66A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Single-occurrence modifications ---
    ("ecoli23S", 2498): ("Cm",    "2\u2032-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D",     "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm",    "2\u2032-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um",    "2\u2032-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C",  "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G",   "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A",   "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Psi", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U",   "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm",  "N4,2\u2032-O-dimethylcytidine"),
}

# Build set of pipeline positions (biological_pos - offset)
KNOWN_MOD_SET_BIO = set(KNOWN_MODIFICATIONS.keys())
KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_PIPELINE_SET = set(KNOWN_MOD_PIPELINE.keys())

# Summary
mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"Modification types: {len(mod_counts)}")
print(f"Position offset: +{POSITION_OFFSET} (pipeline → biological)\n")
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f"  {mod_type:<8s} {count:3d}  {full}")

## 2. Run f5c Eventalign & Parse Signals (Once)

Run f5c eventalign **once** per contig to get signal-level alignments,
then parse the TSV into per-position signal dictionaries.
These signals are reused across all DTW parameter combinations.

In [ ]:
from baleen.eventalign._bam import (
    get_contig_stats, filter_contigs, validate_bam,
    split_bam_contig,
)
from baleen.eventalign._f5c import check_f5c, index_fastq_blow5, index_blow5, run_eventalign
from baleen.eventalign._signal import (
    group_signals_by_position,
    get_common_positions,
    extract_signals_for_dtw,
    extract_signals_for_dtw_padded,
)

MIN_DEPTH = 15

# Step 1: Check f5c
f5c_version = check_f5c()
print(f"f5c version: {f5c_version}")

# Step 2: Index
print("Indexing...")
index_fastq_blow5(NATIVE_FASTQ, NATIVE_BLOW5)
index_fastq_blow5(IVT_FASTQ, IVT_BLOW5)
index_blow5(NATIVE_BLOW5)
index_blow5(IVT_BLOW5)

# Step 3: Contig filtering
validate_bam(NATIVE_BAM)
validate_bam(IVT_BAM)
native_stats = get_contig_stats(NATIVE_BAM)
ivt_stats = get_contig_stats(IVT_BAM)
passed_contigs, filter_results = filter_contigs(native_stats, ivt_stats, min_depth=float(MIN_DEPTH))
print(f"Contigs passing filter: {passed_contigs}")

In [ ]:
import tempfile, shutil

# Step 4: Run f5c eventalign per contig and parse signals
# We keep the parsed signal dicts in memory — no need to re-run f5c.

contig_signals: dict[str, dict] = {}  # contig → {native_by_pos, ivt_by_pos, common_positions}

tmp_root = Path(tempfile.mkdtemp(prefix="baleen-bench-"))
print(f"Temp dir: {tmp_root}")

try:
    for contig in passed_contigs:
        print(f"\n{'='*50}")
        print(f"Processing contig: {contig}")
        contig_tmp = tmp_root / contig
        contig_tmp.mkdir(parents=True, exist_ok=True)

        # Split BAMs
        native_contig_bam = split_bam_contig(NATIVE_BAM, contig, contig_tmp / "native")
        ivt_contig_bam = split_bam_contig(IVT_BAM, contig, contig_tmp / "ivt")

        # Run eventalign
        native_tsv = contig_tmp / "native.eventalign.tsv"
        ivt_tsv = contig_tmp / "ivt.eventalign.tsv"

        print("  Running f5c eventalign (native)...")
        run_eventalign(native_contig_bam, REF_FASTA, NATIVE_FASTQ, NATIVE_BLOW5, native_tsv, rna=True)
        print("  Running f5c eventalign (IVT)...")
        run_eventalign(ivt_contig_bam, REF_FASTA, IVT_FASTQ, IVT_BLOW5, ivt_tsv, rna=True)

        # Parse signals
        native_by_pos = group_signals_by_position(native_tsv)
        ivt_by_pos = group_signals_by_position(ivt_tsv)
        common = get_common_positions(native_by_pos, ivt_by_pos)

        contig_signals[contig] = {
            "native_by_pos": native_by_pos,
            "ivt_by_pos": ivt_by_pos,
            "common_positions": common,
        }
        print(f"  → {len(common)} common positions")
        # Check which known mods fall in the pipeline positions
        found_mods = [(contig, p) for p in common if (contig, p) in KNOWN_MOD_PIPELINE_SET]
        print(f"  → {len(found_mods)} known modification sites found in pipeline positions")
        for c, p in found_mods:
            bio_pos = p + POSITION_OFFSET
            mod_short = KNOWN_MOD_PIPELINE[(c, p)][0]
            print(f"    pos {p} (bio {bio_pos}) → {mod_short}")
finally:
    shutil.rmtree(tmp_root, ignore_errors=True)
    print(f"\nCleaned up temp dir: {tmp_root}")

print(f"\nSignals cached for {len(contig_signals)} contigs.")

## 3. DTW Parameter Grid & Distance Matrix Computation

Compute pairwise DTW distance matrices for every combination of:
- `padding` ∈ {0, 1, 2}
- `(use_open_start, use_open_end)` ∈ {(F,F), (T,F), (F,T), (T,T)}

In [ ]:
from baleen.eventalign._pipeline import PositionResult, ContigResult
from baleen._cuda_dtw import dtw_pairwise_varlen, CUDA_AVAILABLE

# DTW parameter grid
PADDING_VALUES = [0, 1, 2]
OPEN_BOUNDARY_COMBOS = [
    (False, False),
    (True, False),
    (False, True),
    (True, True),
]

DTW_GRID = list(product(PADDING_VALUES, OPEN_BOUNDARY_COMBOS))
print(f"DTW parameter grid: {len(DTW_GRID)} combinations")
for pad, (os, oe) in DTW_GRID:
    print(f"  padding={pad}  open_start={os}  open_end={oe}")

print(f"\nCUDA available: {CUDA_AVAILABLE}")

In [ ]:
def compute_dtw_for_config(
    contig: str,
    sigs: dict,
    padding: int,
    use_open_start: bool,
    use_open_end: bool,
) -> dict[int, PositionResult]:
    """Compute pairwise DTW distance matrices for one contig + one param config."""
    native_by_pos = sigs["native_by_pos"]
    ivt_by_pos = sigs["ivt_by_pos"]
    common = sigs["common_positions"]

    position_results: dict[int, PositionResult] = {}
    for pos in common:
        # Extract signals with or without padding
        if padding > 0:
            native_names, native_sigs = extract_signals_for_dtw_padded(native_by_pos, pos, padding)
            ivt_names, ivt_sigs = extract_signals_for_dtw_padded(ivt_by_pos, pos, padding)
        else:
            native_names, native_sigs = extract_signals_for_dtw(native_by_pos[pos])
            ivt_names, ivt_sigs = extract_signals_for_dtw(ivt_by_pos[pos])

        if not native_sigs or not ivt_sigs:
            continue

        # Filter out empty signals
        valid_native = [(n, s) for n, s in zip(native_names, native_sigs) if len(s) > 0]
        valid_ivt = [(n, s) for n, s in zip(ivt_names, ivt_sigs) if len(s) > 0]
        if not valid_native or not valid_ivt:
            continue
        native_names = [n for n, _ in valid_native]
        native_sigs = [s for _, s in valid_native]
        ivt_names = [n for n, _ in valid_ivt]
        ivt_sigs = [s for _, s in valid_ivt]

        all_sigs = native_sigs + ivt_sigs
        matrix = dtw_pairwise_varlen(
            all_sigs,
            use_open_start=use_open_start,
            use_open_end=use_open_end,
            use_cuda=None,  # auto-detect
        )

        kmer = native_by_pos[pos].reference_kmer
        position_results[pos] = PositionResult(
            position=pos,
            reference_kmer=kmer,
            n_native_reads=len(native_names),
            n_ivt_reads=len(ivt_names),
            native_read_names=native_names,
            ivt_read_names=ivt_names,
            distance_matrix=matrix,
        )

    return position_results

In [ ]:
# Run DTW grid — this is the expensive step
# results_grid[(padding, open_start, open_end)][contig] → dict[pos, PositionResult]

results_grid: dict[tuple, dict[str, dict[int, PositionResult]]] = {}

total_configs = len(DTW_GRID)
t0_grid = time.time()

for cfg_idx, (padding, (open_start, open_end)) in enumerate(DTW_GRID, 1):
    config_key = (padding, open_start, open_end)
    print(f"\n[{cfg_idx}/{total_configs}] padding={padding} open_start={open_start} open_end={open_end}")
    t0 = time.time()

    contig_results: dict[str, dict[int, PositionResult]] = {}
    for contig, sigs in contig_signals.items():
        pos_results = compute_dtw_for_config(contig, sigs, padding, open_start, open_end)
        contig_results[contig] = pos_results
        print(f"  {contig}: {len(pos_results)} positions")

    results_grid[config_key] = contig_results
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s")

total_elapsed = time.time() - t0_grid
print(f"\nDTW grid complete: {total_configs} configs in {total_elapsed:.1f}s")

## 4. Run Probability Methods on Each DTW Config

For each DTW parameter combination, run all 5 probability methods:
- **Family A** (per-position independent): distance_to_ivt, knn_ivt_purity, mds_gmm
- **Family B** (contig-level hierarchical): V2 (mixture raw), V3 (HMM smoothed)

In [ ]:
from baleen.eventalign import (
    AlgorithmName,
    compute_modification_probabilities,
    compute_sequential_modification_probabilities,
    ContigResult,
)

# Method names for the benchmark
METHOD_NAMES = [
    "dist_to_ivt",
    "knn_purity",
    "mds_gmm",
    "hier_V2_raw",
    "hier_V3_hmm",
]


def collect_predictions_for_config(
    config_key: tuple,
    contig_pos_results: dict[str, dict[int, PositionResult]],
) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Run all 5 methods and return {method_name: (y_true, y_score)} arrays.

    Ground truth:
    - At known-mod pipeline positions: native reads = 1, IVT reads = 0
    - At all other positions: all reads = 0
    """
    # Accumulators per method
    all_y_true: dict[str, list[float]] = {m: [] for m in METHOD_NAMES}
    all_y_score: dict[str, list[float]] = {m: [] for m in METHOD_NAMES}

    for contig, pos_results in contig_pos_results.items():
        if not pos_results:
            continue

        # ---------- Family A: per-position ----------
        for pos, pr in pos_results.items():
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            # Ground truth: native reads first, then IVT
            if is_mod:
                y_true_pos = np.concatenate([
                    np.ones(n_nat),   # native at mod site → modified
                    np.zeros(n_ivt),  # IVT at mod site → unmodified
                ])
            else:
                y_true_pos = np.zeros(n_nat + n_ivt)  # all unmodified

            # Run Family A algorithms
            try:
                probs_dict = compute_modification_probabilities(
                    pr.distance_matrix, n_nat, n_ivt, position=pos,
                )
            except Exception:
                continue

            for alg, method_name in [
                (AlgorithmName.DISTANCE_TO_IVT, "dist_to_ivt"),
                (AlgorithmName.KNN_IVT_PURITY, "knn_purity"),
                (AlgorithmName.MDS_GMM, "mds_gmm"),
            ]:
                mp = probs_dict[alg]
                all_y_true[method_name].append(y_true_pos)
                all_y_score[method_name].append(mp.probabilities)

        # ---------- Family B: contig-level hierarchical ----------
        # Build a ContigResult to pass to hierarchical pipeline
        depths = contig_signals.get(contig, {})
        cr = ContigResult(
            contig=contig,
            native_depth=0.0,  # not used by hierarchical
            ivt_depth=0.0,
            positions=pos_results,
        )

        try:
            hres = compute_sequential_modification_probabilities(cr)
        except Exception:
            continue

        for pos, ps in hres.position_stats.items():
            if pos not in pos_results:
                continue
            pr = pos_results[pos]
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            if is_mod:
                y_true_pos = np.concatenate([np.ones(n_nat), np.zeros(n_ivt)])
            else:
                y_true_pos = np.zeros(n_nat + n_ivt)

            # V2 raw
            all_y_true["hier_V2_raw"].append(y_true_pos)
            all_y_score["hier_V2_raw"].append(ps.p_mod_raw)

            # V3 HMM
            all_y_true["hier_V3_hmm"].append(y_true_pos)
            all_y_score["hier_V3_hmm"].append(ps.p_mod_hmm)

    # Concatenate
    result = {}
    for m in METHOD_NAMES:
        if all_y_true[m]:
            yt = np.concatenate(all_y_true[m])
            ys = np.concatenate(all_y_score[m])
            # Drop NaN scores
            valid = ~np.isnan(ys)
            result[m] = (yt[valid], ys[valid])
        else:
            result[m] = (np.array([]), np.array([]))

    return result

In [ ]:
# Run all methods across the DTW grid
# predictions[(padding, open_start, open_end)][method_name] = (y_true, y_score)

predictions: dict[tuple, dict[str, tuple[np.ndarray, np.ndarray]]] = {}

t0_all = time.time()
for cfg_idx, (padding, (open_start, open_end)) in enumerate(DTW_GRID, 1):
    config_key = (padding, open_start, open_end)
    print(f"[{cfg_idx}/{len(DTW_GRID)}] pad={padding} os={open_start} oe={open_end} ... ", end="")
    t0 = time.time()

    preds = collect_predictions_for_config(config_key, results_grid[config_key])
    predictions[config_key] = preds

    # Quick summary
    for m in METHOD_NAMES:
        yt, ys = preds[m]
        n_pos = int(yt.sum())
        print(f"{m}:{len(yt)}({n_pos}+) ", end="")
    print(f"  [{time.time() - t0:.1f}s]")

print(f"\nAll predictions collected in {time.time() - t0_all:.1f}s")

## 5. Compute Metrics

For each of the 60 configurations, compute:
- **AUROC** — area under ROC curve
- **AUPRC** — area under precision-recall curve
- **F1** — at optimal threshold (maximizing F1)
- **Precision / Recall** at optimal F1 threshold

In [ ]:
def compute_metrics(y_true: np.ndarray, y_score: np.ndarray) -> dict:
    """Compute classification metrics for a single (y_true, y_score) pair."""
    if len(y_true) == 0 or y_true.sum() == 0 or y_true.sum() == len(y_true):
        return {
            "auroc": np.nan, "auprc": np.nan, "f1": np.nan,
            "precision": np.nan, "recall": np.nan, "threshold": np.nan,
            "n_total": len(y_true), "n_pos": int(y_true.sum()),
            "n_neg": int(len(y_true) - y_true.sum()),
        }

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)

    # Find optimal F1 threshold
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)
    # precision_recall_curve returns n+1 precisions/recalls but n thresholds
    f1s = np.where(
        (precisions[:-1] + recalls[:-1]) > 0,
        2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1]),
        0.0,
    )
    best_idx = np.argmax(f1s)
    best_f1 = float(f1s[best_idx])
    best_threshold = float(thresholds[best_idx])
    best_precision = float(precisions[:-1][best_idx])
    best_recall = float(recalls[:-1][best_idx])

    return {
        "auroc": auroc,
        "auprc": auprc,
        "f1": best_f1,
        "precision": best_precision,
        "recall": best_recall,
        "threshold": best_threshold,
        "n_total": len(y_true),
        "n_pos": int(y_true.sum()),
        "n_neg": int(len(y_true) - y_true.sum()),
    }

In [ ]:
# Build the full benchmark DataFrame
rows = []
for (padding, open_start, open_end), preds in predictions.items():
    for method, (y_true, y_score) in preds.items():
        metrics = compute_metrics(y_true, y_score)
        rows.append({
            "padding": padding,
            "open_start": open_start,
            "open_end": open_end,
            "method": method,
            **metrics,
        })

df_bench = pd.DataFrame(rows)

# Sort by AUROC descending
df_bench = df_bench.sort_values("auroc", ascending=False).reset_index(drop=True)

print(f"Benchmark results: {len(df_bench)} configurations")
print(f"\nTop 10 by AUROC:\n")
display_cols = ["padding", "open_start", "open_end", "method",
                "auroc", "auprc", "f1", "precision", "recall", "n_pos", "n_neg"]
df_bench[display_cols].head(10)

In [ ]:
# Full table
print("\nFull benchmark results (sorted by AUROC):\n")
df_bench[display_cols].to_string(index=True)
df_bench[display_cols]

## 5b. Position-Level Evaluation

The per-read evaluation above is dominated by true negatives (class imbalance
~99:1).  A **position-level** view is more aligned with the actual use case:
"which positions are modified?"

For each position we aggregate native read probabilities into a single
position-level score (median, and fraction above 0.5), then compute
AUROC/AUPRC over positions (43 modified vs hundreds unmodified).

In [ ]:
def collect_position_level_predictions(
    config_key: tuple,
    contig_pos_results: dict[str, dict[int, PositionResult]],
) -> dict[str, dict[str, tuple[np.ndarray, np.ndarray]]]:
    """Aggregate per-read probabilities to position-level scores.

    For each position, compute two aggregation strategies:
      - 'median': median P(mod) across native reads
      - 'frac_above_0.5': fraction of native reads with P(mod) > 0.5

    Returns: {agg_strategy: {method_name: (y_true_positions, y_score_positions)}}
    where y_true = 1 for known modification positions, 0 otherwise.
    """
    AGG_STRATEGIES = ["median", "frac_above_0.5"]
    # Accumulators: agg → method → (y_true list, y_score list)
    acc = {
        agg: {m: ([], []) for m in METHOD_NAMES}
        for agg in AGG_STRATEGIES
    }

    for contig, pos_results in contig_pos_results.items():
        if not pos_results:
            continue

        # --- Family A: per-position algorithms ---
        for pos, pr in pos_results.items():
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            try:
                probs_dict = compute_modification_probabilities(
                    pr.distance_matrix, n_nat, n_ivt, position=pos,
                )
            except Exception:
                continue

            for alg, method_name in [
                (AlgorithmName.DISTANCE_TO_IVT, "dist_to_ivt"),
                (AlgorithmName.KNN_IVT_PURITY, "knn_purity"),
                (AlgorithmName.MDS_GMM, "mds_gmm"),
            ]:
                native_probs = probs_dict[alg].native_probabilities
                if len(native_probs) == 0:
                    continue

                median_score = float(np.median(native_probs))
                frac_score = float(np.mean(native_probs > 0.5))
                label = 1.0 if is_mod else 0.0

                acc["median"][method_name][0].append(label)
                acc["median"][method_name][1].append(median_score)
                acc["frac_above_0.5"][method_name][0].append(label)
                acc["frac_above_0.5"][method_name][1].append(frac_score)

        # --- Family B: contig-level hierarchical ---
        cr = ContigResult(
            contig=contig, native_depth=0.0, ivt_depth=0.0,
            positions=pos_results,
        )
        try:
            hres = compute_sequential_modification_probabilities(cr)
        except Exception:
            continue

        for pos, ps in hres.position_stats.items():
            if pos not in pos_results:
                continue
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            label = 1.0 if is_mod else 0.0

            for method_name, arr in [
                ("hier_V2_raw", ps.p_mod_raw[:ps.n_native]),
                ("hier_V3_hmm", ps.p_mod_hmm[:ps.n_native]),
            ]:
                valid = arr[~np.isnan(arr)]
                if len(valid) == 0:
                    continue

                median_score = float(np.median(valid))
                frac_score = float(np.mean(valid > 0.5))

                acc["median"][method_name][0].append(label)
                acc["median"][method_name][1].append(median_score)
                acc["frac_above_0.5"][method_name][0].append(label)
                acc["frac_above_0.5"][method_name][1].append(frac_score)

    # Convert to arrays
    result = {}
    for agg in AGG_STRATEGIES:
        result[agg] = {}
        for m in METHOD_NAMES:
            yt = np.array(acc[agg][m][0])
            ys = np.array(acc[agg][m][1])
            valid = ~np.isnan(ys)
            result[agg][m] = (yt[valid], ys[valid])

    return result

In [ ]:
# Run position-level evaluation across DTW grid
pos_level_predictions: dict[tuple, dict[str, dict[str, tuple[np.ndarray, np.ndarray]]]] = {}

t0_pos = time.time()
for cfg_idx, (padding, (open_start, open_end)) in enumerate(DTW_GRID, 1):
    config_key = (padding, open_start, open_end)
    print(f"[{cfg_idx}/{len(DTW_GRID)}] pad={padding} os={open_start} oe={open_end} ... ", end="")

    pos_preds = collect_position_level_predictions(config_key, results_grid[config_key])
    pos_level_predictions[config_key] = pos_preds

    # Quick summary
    for agg in ["median", "frac_above_0.5"]:
        for m in METHOD_NAMES[:2]:  # just show first two
            yt, ys = pos_preds[agg][m]
            n_pos = int(yt.sum())
            n_neg = int(len(yt) - yt.sum())
            print(f"{agg}/{m}:{n_pos}+/{n_neg}- ", end="")
    print(f"  [{time.time() - t0_pos:.1f}s]")

print(f"\nPosition-level predictions collected in {time.time() - t0_pos:.1f}s")

In [ ]:
# Build position-level benchmark DataFrame
pos_rows = []
for (padding, open_start, open_end), agg_preds in pos_level_predictions.items():
    for agg_strategy, method_preds in agg_preds.items():
        for method, (y_true, y_score) in method_preds.items():
            metrics = compute_metrics(y_true, y_score)
            pos_rows.append({
                "padding": padding,
                "open_start": open_start,
                "open_end": open_end,
                "method": method,
                "aggregation": agg_strategy,
                **metrics,
            })

df_pos_bench = pd.DataFrame(pos_rows)
df_pos_bench = df_pos_bench.sort_values("auprc", ascending=False).reset_index(drop=True)

print(f"Position-level benchmark: {len(df_pos_bench)} configurations")
print(f"\nTop 15 by AUPRC (position-level):\n")
pos_display_cols = ["padding", "open_start", "open_end", "method", "aggregation",
                    "auroc", "auprc", "f1", "precision", "recall", "n_pos", "n_neg"]
df_pos_bench[pos_display_cols].head(15)

In [ ]:
# Position-level comparison: bar charts for best config per method×aggregation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, agg in zip(axes, ["median", "frac_above_0.5"]):
    df_agg = df_pos_bench[df_pos_bench["aggregation"] == agg]
    best = df_agg.loc[df_agg.groupby("method")["auprc"].idxmax()]
    best = best.sort_values("auprc", ascending=True)

    y_pos = range(len(best))
    bars_auroc = ax.barh(
        [y - 0.15 for y in y_pos], best["auroc"].values, height=0.3,
        color="#5B9BD5", label="AUROC", edgecolor="black", linewidth=0.3,
    )
    bars_auprc = ax.barh(
        [y + 0.15 for y in y_pos], best["auprc"].values, height=0.3,
        color="#ED7D31", label="AUPRC", edgecolor="black", linewidth=0.3,
    )

    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([
        f"{r['method']}\npad={int(r['padding'])} os={r['open_start']} oe={r['open_end']}"
        for _, r in best.iterrows()
    ], fontsize=8)
    ax.set_xlabel("Score")
    ax.set_xlim(0, 1.05)
    ax.set_title(f"Position-Level Metrics (agg={agg})", fontweight="bold")
    ax.legend(fontsize=8)

    for bar, val in zip(bars_auroc, best["auroc"].values):
        if not np.isnan(val):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=7)
    for bar, val in zip(bars_auprc, best["auprc"].values):
        if not np.isnan(val):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=7)

fig.suptitle("Position-Level Evaluation (best DTW config per method)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Print comparison of per-read vs position-level AUPRC
print("\nPer-read vs Position-level AUPRC comparison (best config per method):")
print(f"{'Method':20s} {'Per-read AUPRC':>15s} {'Pos median AUPRC':>17s} {'Pos frac AUPRC':>15s}")
print("-" * 70)
for method in METHOD_NAMES:
    pr_row = df_bench[df_bench["method"] == method].sort_values("auprc", ascending=False).head(1)
    med_row = df_pos_bench[(df_pos_bench["method"] == method) &
                           (df_pos_bench["aggregation"] == "median")].sort_values("auprc", ascending=False).head(1)
    frac_row = df_pos_bench[(df_pos_bench["method"] == method) &
                            (df_pos_bench["aggregation"] == "frac_above_0.5")].sort_values("auprc", ascending=False).head(1)

    pr_val = f"{pr_row.iloc[0]['auprc']:.4f}" if len(pr_row) > 0 else "N/A"
    med_val = f"{med_row.iloc[0]['auprc']:.4f}" if len(med_row) > 0 else "N/A"
    frac_val = f"{frac_row.iloc[0]['auprc']:.4f}" if len(frac_row) > 0 else "N/A"
    print(f"{method:20s} {pr_val:>15s} {med_val:>17s} {frac_val:>15s}")

## 6. Visualization

### 6a. AUROC Heatmaps per Method

Rows = padding, Columns = open boundary combo. One heatmap per method.

In [ ]:
fig, axes = plt.subplots(1, len(METHOD_NAMES), figsize=(4.5 * len(METHOD_NAMES), 4), squeeze=False)
fig.suptitle("AUROC by DTW Parameters × Method", fontsize=14, fontweight="bold", y=1.05)

boundary_labels = ["(F,F)", "(T,F)", "(F,T)", "(T,T)"]
padding_labels = [str(p) for p in PADDING_VALUES]

for col_idx, method in enumerate(METHOD_NAMES):
    ax = axes[0][col_idx]
    df_m = df_bench[df_bench["method"] == method]

    # Build heatmap matrix: rows=padding, cols=boundary combo
    heat = np.full((len(PADDING_VALUES), len(OPEN_BOUNDARY_COMBOS)), np.nan)
    for i, pad in enumerate(PADDING_VALUES):
        for j, (os, oe) in enumerate(OPEN_BOUNDARY_COMBOS):
            row = df_m[(df_m["padding"] == pad) &
                       (df_m["open_start"] == os) &
                       (df_m["open_end"] == oe)]
            if len(row) > 0:
                heat[i, j] = row.iloc[0]["auroc"]

    vmin = np.nanmin(df_bench["auroc"]) if not df_bench["auroc"].isna().all() else 0
    vmax = np.nanmax(df_bench["auroc"]) if not df_bench["auroc"].isna().all() else 1

    im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(boundary_labels)))
    ax.set_xticklabels(boundary_labels, fontsize=8, rotation=45)
    ax.set_yticks(range(len(padding_labels)))
    ax.set_yticklabels(padding_labels)
    ax.set_xlabel("(open_start, open_end)")
    if col_idx == 0:
        ax.set_ylabel("padding")
    ax.set_title(method, fontsize=10, fontweight="bold")

    # Annotate cells
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            if not np.isnan(heat[i, j]):
                ax.text(j, i, f"{heat[i,j]:.3f}", ha="center", va="center",
                        fontsize=8, color="black")

fig.colorbar(im, ax=axes[0].tolist(), label="AUROC", shrink=0.8)
plt.tight_layout()
plt.show()

### 6b. AUPRC Heatmaps per Method

In [ ]:
fig, axes = plt.subplots(1, len(METHOD_NAMES), figsize=(4.5 * len(METHOD_NAMES), 4), squeeze=False)
fig.suptitle("AUPRC by DTW Parameters × Method", fontsize=14, fontweight="bold", y=1.05)

for col_idx, method in enumerate(METHOD_NAMES):
    ax = axes[0][col_idx]
    df_m = df_bench[df_bench["method"] == method]

    heat = np.full((len(PADDING_VALUES), len(OPEN_BOUNDARY_COMBOS)), np.nan)
    for i, pad in enumerate(PADDING_VALUES):
        for j, (os, oe) in enumerate(OPEN_BOUNDARY_COMBOS):
            row = df_m[(df_m["padding"] == pad) &
                       (df_m["open_start"] == os) &
                       (df_m["open_end"] == oe)]
            if len(row) > 0:
                heat[i, j] = row.iloc[0]["auprc"]

    vmin = np.nanmin(df_bench["auprc"]) if not df_bench["auprc"].isna().all() else 0
    vmax = np.nanmax(df_bench["auprc"]) if not df_bench["auprc"].isna().all() else 1

    im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(boundary_labels)))
    ax.set_xticklabels(boundary_labels, fontsize=8, rotation=45)
    ax.set_yticks(range(len(padding_labels)))
    ax.set_yticklabels(padding_labels)
    ax.set_xlabel("(open_start, open_end)")
    if col_idx == 0:
        ax.set_ylabel("padding")
    ax.set_title(method, fontsize=10, fontweight="bold")

    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            if not np.isnan(heat[i, j]):
                ax.text(j, i, f"{heat[i,j]:.3f}", ha="center", va="center",
                        fontsize=8, color="black")

fig.colorbar(im, ax=axes[0].tolist(), label="AUPRC", shrink=0.8)
plt.tight_layout()
plt.show()

### 6c. Bar Chart — Best DTW Config per Method

For each method, show the best DTW config and its metrics.

In [ ]:
# Best config per method (by AUROC)
best_per_method = df_bench.loc[df_bench.groupby("method")["auroc"].idxmax()]
best_per_method = best_per_method.sort_values("auroc", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metric_names = ["auroc", "auprc", "f1"]
metric_labels = ["AUROC", "AUPRC", "F1 (optimal threshold)"]
colors = ["#5B9BD5", "#70AD47", "#FFC000", "#ED7D31", "#A855F7"]

for ax, metric, label in zip(axes, metric_names, metric_labels):
    bars = ax.barh(
        range(len(best_per_method)),
        best_per_method[metric].values,
        color=colors[:len(best_per_method)],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_yticks(range(len(best_per_method)))
    ylabels = []
    for _, r in best_per_method.iterrows():
        ylabels.append(f"{r['method']}\npad={int(r['padding'])} os={r['open_start']} oe={r['open_end']}")
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.set_xlabel(label)
    ax.set_xlim(0, 1.05)
    ax.set_title(label, fontweight="bold")

    # Annotate bars
    for bar, val in zip(bars, best_per_method[metric].values):
        if not np.isnan(val):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=8)

fig.suptitle("Best DTW Config per Method", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nBest configs:")
best_per_method[display_cols]

### 6d. ROC Curves — Best Config per Method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
method_colors = dict(zip(METHOD_NAMES, ["#5B9BD5", "#70AD47", "#FFC000", "#ED7D31", "#A855F7"]))

# ROC curves
ax = axes[0]
for _, row in best_per_method.iterrows():
    method = row["method"]
    config_key = (int(row["padding"]), row["open_start"], row["open_end"])
    y_true, y_score = predictions[config_key][method]
    if len(y_true) > 0 and y_true.sum() > 0:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, color=method_colors[method], linewidth=2,
                label=f"{method} (AUROC={row['auroc']:.3f})")

ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Best Config per Method", fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

# PR curves
ax = axes[1]
for _, row in best_per_method.iterrows():
    method = row["method"]
    config_key = (int(row["padding"]), row["open_start"], row["open_end"])
    y_true, y_score = predictions[config_key][method]
    if len(y_true) > 0 and y_true.sum() > 0:
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        ax.plot(rec, prec, color=method_colors[method], linewidth=2,
                label=f"{method} (AUPRC={row['auprc']:.3f})")

# Baseline = prevalence
if len(best_per_method) > 0:
    first_row = best_per_method.iloc[0]
    prevalence = first_row["n_pos"] / first_row["n_total"] if first_row["n_total"] > 0 else 0
    ax.axhline(prevalence, color="gray", linestyle="--", linewidth=1, label=f"baseline ({prevalence:.3f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Best Config per Method", fontweight="bold")
ax.legend(fontsize=8, loc="upper right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6e. Method Comparison Across All DTW Configs

Box plot showing the distribution of AUROC/AUPRC for each method across
all 12 DTW parameter combinations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, label in zip(axes, ["auroc", "auprc", "f1"],
                              ["AUROC", "AUPRC", "F1"]):
    data_per_method = []
    labels = []
    for method in METHOD_NAMES:
        vals = df_bench[df_bench["method"] == method][metric].dropna().values
        if len(vals) > 0:
            data_per_method.append(vals)
            labels.append(method)

    if data_per_method:
        bp = ax.boxplot(data_per_method, labels=labels, patch_artist=True, widths=0.5)
        for patch, color in zip(bp["boxes"], colors[:len(data_per_method)]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

    ax.set_ylabel(label)
    ax.set_title(f"{label} Distribution Across DTW Configs", fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Per-Modification-Type Breakdown

Evaluate how well each method detects each modification type (Psi, m5C,
m6A, etc.) using the **best overall DTW config** per method.

In [ ]:
def collect_per_mod_type_predictions(
    config_key: tuple,
    contig_pos_results: dict[str, dict[int, PositionResult]],
) -> dict[str, dict[str, tuple[np.ndarray, np.ndarray]]]:
    """Like collect_predictions_for_config but partitioned by modification type.

    Returns: {mod_type: {method_name: (y_true, y_score)}}
    where mod_type includes all known types plus "unmodified" for negative sites.
    """
    # Accumulators: mod_type → method → (y_true_list, y_score_list)
    acc: dict[str, dict[str, tuple[list, list]]] = defaultdict(
        lambda: {m: ([], []) for m in METHOD_NAMES}
    )

    for contig, pos_results in contig_pos_results.items():
        if not pos_results:
            continue

        # Family A
        for pos, pr in pos_results.items():
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            if is_mod:
                mod_type = KNOWN_MOD_PIPELINE[(contig, pos)][0]
                y_true_nat = np.ones(n_nat)
                y_true_ivt = np.zeros(n_ivt)
            else:
                mod_type = "unmodified"
                y_true_nat = np.zeros(n_nat)
                y_true_ivt = np.zeros(n_ivt)

            y_true_pos = np.concatenate([y_true_nat, y_true_ivt])

            try:
                probs_dict = compute_modification_probabilities(
                    pr.distance_matrix, n_nat, n_ivt, position=pos,
                )
            except Exception:
                continue

            for alg, method_name in [
                (AlgorithmName.DISTANCE_TO_IVT, "dist_to_ivt"),
                (AlgorithmName.KNN_IVT_PURITY, "knn_purity"),
                (AlgorithmName.MDS_GMM, "mds_gmm"),
            ]:
                mp = probs_dict[alg]
                acc[mod_type][method_name][0].append(y_true_pos)
                acc[mod_type][method_name][1].append(mp.probabilities)

        # Family B
        cr = ContigResult(
            contig=contig, native_depth=0.0, ivt_depth=0.0,
            positions=pos_results,
        )
        try:
            hres = compute_sequential_modification_probabilities(cr)
        except Exception:
            continue

        for pos, ps in hres.position_stats.items():
            if pos not in pos_results:
                continue
            pr = pos_results[pos]
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            if is_mod:
                mod_type = KNOWN_MOD_PIPELINE[(contig, pos)][0]
                y_true_pos = np.concatenate([np.ones(n_nat), np.zeros(n_ivt)])
            else:
                mod_type = "unmodified"
                y_true_pos = np.zeros(n_nat + n_ivt)

            acc[mod_type]["hier_V2_raw"][0].append(y_true_pos)
            acc[mod_type]["hier_V2_raw"][1].append(ps.p_mod_raw)
            acc[mod_type]["hier_V3_hmm"][0].append(y_true_pos)
            acc[mod_type]["hier_V3_hmm"][1].append(ps.p_mod_hmm)

    # Concatenate
    result: dict[str, dict[str, tuple[np.ndarray, np.ndarray]]] = {}
    for mod_type, methods in acc.items():
        result[mod_type] = {}
        for m, (yt_list, ys_list) in methods.items():
            if yt_list:
                yt = np.concatenate(yt_list)
                ys = np.concatenate(ys_list)
                valid = ~np.isnan(ys)
                result[mod_type][m] = (yt[valid], ys[valid])
            else:
                result[mod_type][m] = (np.array([]), np.array([]))

    return result

In [ ]:
# For each method, use its best DTW config to do per-mod-type analysis
mod_type_rows = []

for _, best_row in best_per_method.iterrows():
    method = best_row["method"]
    config_key = (int(best_row["padding"]), best_row["open_start"], best_row["open_end"])
    contig_pos_results = results_grid[config_key]

    per_mod = collect_per_mod_type_predictions(config_key, contig_pos_results)

    for mod_type, method_preds in per_mod.items():
        y_true, y_score = method_preds[method]
        if len(y_true) == 0:
            continue

        # For "unmodified" positions, all labels are 0 → skip AUROC
        if mod_type == "unmodified":
            # Report mean score as false positive indicator
            mod_type_rows.append({
                "method": method,
                "mod_type": mod_type,
                "n_samples": len(y_true),
                "n_pos": 0,
                "mean_score": float(np.mean(y_score)),
                "auroc": np.nan,
                "auprc": np.nan,
                "f1": np.nan,
            })
        else:
            metrics = compute_metrics(y_true, y_score)
            mod_type_rows.append({
                "method": method,
                "mod_type": mod_type,
                "n_samples": len(y_true),
                "n_pos": int(y_true.sum()),
                "mean_score": float(np.mean(y_score[y_true == 1])) if y_true.sum() > 0 else np.nan,
                "auroc": metrics["auroc"],
                "auprc": metrics["auprc"],
                "f1": metrics["f1"],
            })

df_mod_type = pd.DataFrame(mod_type_rows)
print(f"Per-modification-type breakdown: {len(df_mod_type)} entries\n")
df_mod_type

In [ ]:
# Heatmap: mod_type × method, showing AUROC
mod_types_with_pos = sorted([m for m in df_mod_type["mod_type"].unique() if m != "unmodified"])

if mod_types_with_pos:
    heat_data = np.full((len(mod_types_with_pos), len(METHOD_NAMES)), np.nan)
    for i, mt in enumerate(mod_types_with_pos):
        for j, method in enumerate(METHOD_NAMES):
            row = df_mod_type[(df_mod_type["mod_type"] == mt) & (df_mod_type["method"] == method)]
            if len(row) > 0:
                heat_data[i, j] = row.iloc[0]["auroc"]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.5 * len(mod_types_with_pos))))
    im = ax.imshow(heat_data, aspect="auto", cmap="RdYlGn", vmin=0.4, vmax=1.0)
    ax.set_xticks(range(len(METHOD_NAMES)))
    ax.set_xticklabels(METHOD_NAMES, fontsize=9, rotation=30, ha="right")
    ax.set_yticks(range(len(mod_types_with_pos)))
    ax.set_yticklabels(mod_types_with_pos, fontsize=9)
    ax.set_title("AUROC per Modification Type × Method\n(best DTW config per method)",
                 fontsize=12, fontweight="bold")

    for i in range(heat_data.shape[0]):
        for j in range(heat_data.shape[1]):
            if not np.isnan(heat_data[i, j]):
                ax.text(j, i, f"{heat_data[i,j]:.2f}", ha="center", va="center",
                        fontsize=8, color="black")

    fig.colorbar(im, ax=ax, label="AUROC", shrink=0.8)
    plt.tight_layout()
    plt.show()
else:
    print("No modification types with positive samples found — check position offset.")

### 7b. Per-Mod-Type Mean Probability (True Positives)

How high is the predicted probability for native reads at known modification
sites, broken down by modification type?

In [ ]:
# Filter to only modification types (not "unmodified")
df_mod_only = df_mod_type[df_mod_type["mod_type"] != "unmodified"].copy()

if not df_mod_only.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    mod_types_sorted = df_mod_only.groupby("mod_type")["mean_score"].mean().sort_values(ascending=False).index
    x = np.arange(len(mod_types_sorted))
    width = 0.15

    for i, method in enumerate(METHOD_NAMES):
        vals = []
        for mt in mod_types_sorted:
            row = df_mod_only[(df_mod_only["mod_type"] == mt) & (df_mod_only["method"] == method)]
            vals.append(row.iloc[0]["mean_score"] if len(row) > 0 else 0)
        ax.bar(x + i * width, vals, width, label=method, color=colors[i], edgecolor="black", linewidth=0.3)

    ax.set_xticks(x + width * (len(METHOD_NAMES) - 1) / 2)
    ax.set_xticklabels(mod_types_sorted, fontsize=9, rotation=30, ha="right")
    ax.set_ylabel("Mean P(modified) for true positives")
    ax.set_title("Detection Strength by Modification Type", fontweight="bold")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No per-modification-type data available.")

## 8. Summary & Conclusions

In [ ]:
print("=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)
print(f"\nTotal configurations evaluated: {len(df_bench)}")
print(f"DTW parameter grid: {len(DTW_GRID)} combos (3 padding × 4 boundary)")
print(f"Probability methods: {len(METHOD_NAMES)}")
print(f"Position offset: +{POSITION_OFFSET}")
print()

# Overall best
if not df_bench.empty and not df_bench["auroc"].isna().all():
    best_overall = df_bench.iloc[0]
    print("BEST OVERALL (by AUROC):")
    print(f"  Method:     {best_overall['method']}")
    print(f"  DTW config: padding={int(best_overall['padding'])}, "
          f"open_start={best_overall['open_start']}, open_end={best_overall['open_end']}")
    print(f"  AUROC:      {best_overall['auroc']:.4f}")
    print(f"  AUPRC:      {best_overall['auprc']:.4f}")
    print(f"  F1:         {best_overall['f1']:.4f}")
    print(f"  Precision:  {best_overall['precision']:.4f}")
    print(f"  Recall:     {best_overall['recall']:.4f}")
    print(f"  Threshold:  {best_overall['threshold']:.4f}")
    print(f"  Samples:    {int(best_overall['n_total'])} ({int(best_overall['n_pos'])}+ / {int(best_overall['n_neg'])}-")

print("\n" + "-" * 70)
print("\nBest per method:")
for _, r in best_per_method.iterrows():
    print(f"  {r['method']:15s}  AUROC={r['auroc']:.3f}  AUPRC={r['auprc']:.3f}  "
          f"F1={r['f1']:.3f}  pad={int(r['padding'])} os={r['open_start']} oe={r['open_end']}")

# DTW param impact
print("\n" + "-" * 70)
print("\nDTW parameter impact (mean AUROC across methods):")
print("\n  By padding:")
for pad in PADDING_VALUES:
    mean_auroc = df_bench[df_bench["padding"] == pad]["auroc"].mean()
    print(f"    padding={pad}: {mean_auroc:.4f}")

print("\n  By boundary condition:")
for os, oe in OPEN_BOUNDARY_COMBOS:
    mean_auroc = df_bench[(df_bench["open_start"] == os) & (df_bench["open_end"] == oe)]["auroc"].mean()
    print(f"    open_start={os}, open_end={oe}: {mean_auroc:.4f}")

print("\n" + "=" * 70)

In [ ]:
# Save benchmark results to CSV
csv_path = OUTPUT_DIR / "benchmark_results.csv"
df_bench.to_csv(csv_path, index=False)
print(f"Benchmark results saved to: {csv_path}")

if not df_mod_type.empty:
    mod_csv_path = OUTPUT_DIR / "benchmark_per_mod_type.csv"
    df_mod_type.to_csv(mod_csv_path, index=False)
    print(f"Per-mod-type results saved to: {mod_csv_path}")

# Save position-level results
pos_csv_path = OUTPUT_DIR / "benchmark_position_level.csv"
df_pos_bench.to_csv(pos_csv_path, index=False)
print(f"Position-level results saved to: {pos_csv_path}")

---

**Done.** Results are saved in the output directory as CSV files.
Use `benchmark_results.csv` for the full 60-config comparison and
`benchmark_per_mod_type.csv` for the per-modification-type breakdown.